In [1]:
import sys
import numpy as np
import pybullet as p
import time
import logging
from typing import List, Tuple, Optional, Sequence, Collection, Dict, Any, cast
import random
import json
import ipdb

from predicators.structs import Action, Array, GroundAtom, Object, State, Type, ParameterizedOption
from predicators import utils
from predicators.settings import CFG
from gym.spaces import Box

#Import core environment methods, robot function etc.

from predicators.envs.pybullet_blocks import PyBulletBlocksEnv
from predicators.envs.pybullet_env import PyBulletEnv, create_pybullet_block
from predicators.pybullet_helpers.robots import SingleArmPyBulletRobot
from predicators.pybullet_helpers.robots.mobile_single_arm import MobileSingleArmPyBulletRobot
from predicators.pybullet_helpers.geometry import Pose
from predicators.pybullet_helpers.joint import JointPositions, get_joint_infos, get_joint_positions
from predicators.pybullet_helpers.link import get_link_state, get_link_pose

#Import the functions that are to be tested:

from predicators.pybullet_helpers.motion_planning import run_motion_planning, run_base_motion_planning,\
                                                            run_coordinated_motion_planning
#The pick/place options to be tested are accessed via the env instance
from predicators.pybullet_helpers.controllers import execute_coordinated_path, create_move_end_effector_to_pose_option,\
                                                    create_change_fingers_option, create_move_base_option
#Configure logging for better debugging outputs:
#logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

logging.basicConfig(
    level=logging.WARNING,                    
    format="%(asctime)s %(name)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

pybullet build time: Jan 29 2025 23:16:28


In [2]:
#Defining test configuration, and overriding some default ones:
CFG.pybullet_robot = "fetch_mobile"
CFG.use_gui = True
#Draws helpful debug lines in the workspace.
#NOT SURE WHETHER TO USE THIS. WILL DECIDE AFTER A COUPLE RUNS.
#CFG.pybullet_draw_debug = True
#Initializing with standard size of blocks.
CFG.blocks_block_size = 0.05
CFG.pybullet_birrt_num_iters = 50
CFG.pybullet_birrt_num_attempts = 10
CFG.pybullet_birrt_smooth_amt = 20
CFG.seed = random.randint(0,10000)
#CFG.seed = 12
#Num of PyBullet physics steps per high-level Action in visualize_action_sequence
CFG.pybullet_sim_steps_per_action = 120

In [3]:
#Function to reset robot to a known pose
def reset_robot_fetch_mobile(robot: MobileSingleArmPyBulletRobot,
                             physics_client_id:int,
                             base_pose: Tuple[float, float, float] = (1.35, 0.75, 0.0), # (x,y,theta)
                             arm_joint_angle: Optional[List[float]]=None):
    """
    Resets the robot's base/ teleports it and arm to specified poses.

    """

    robot.move_base_to(base_pose, physics_client_id)
    if arm_joint_angle:
        #Set arm joints only
        robot.set_joints(arm_joint_angle)
    else:
        #robot.initial_joint_positions includes arm and finger joints
        robot.set_joints(robot.initial_joint_positions)
    #Step simulation a bit to allow PyBullet to settle the state.
    for _ in range(10):
        p.stepSimulation(physicsClientId=physics_client_id)


#Fn to create blocks in the env.
def create_test_block(env: PyBulletEnv,
                      pose: Tuple[float, float, float],
                      color: Tuple[float, float, float, float] = (0.8, 0.2, 0.2, 1.0),
                      name_suffix: str = "test") -> int:
    """
    Creates a single block at a specified pose for testing and returns its PyBullet ID.
    """

    #Use the fn defined in utils to create block
    block_id = create_pybullet_block(
        color,
        (CFG.blocks_block_size/2,)*3,
        env._obj_mass,
        env._obj_friction,
        env._default_orn,
        env._physics_client_id
    )
    #Place the block at desired pose.
    p.resetBasePositionAndOrientation(block_id, pose, env._default_orn, physicsClientId=env._physics_client_id)

    return block_id


#Get the list of all bodies except the robot.
#TODO: Need to add logic that saves object/body name
#      which can be used for better debugging with collision.
def get_all_non_robot_bodies(robot_id: int, physics_client_id:int) -> List[int]:
    """
    Gets all PyBullet body IDs in the simulation except for the robot itself.
    These are typically used as collision obstacles.
    """
    all_bodies = [p.getBodyUniqueId(i, physicsClientId=physics_client_id)
                    for i in range(p.getNumBodies(physicsClientId=physics_client_id))]


    return [b for b in all_bodies if b!=robot_id]

In [4]:
#Setup Env.
#Initialize the PyBulletBlocksEnv which sets up PyBullet,
#loads the robot, tables etc.

env = PyBulletBlocksEnv(use_gui=CFG.use_gui)

/home/cloaked04/anaconda3/envs/predicators/lib/python3.10/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


In [5]:
 # Resets the environment to a specific task, getting an initial symbolic state.
# While initial_state_from_env is fetched, the option tests will create their own
# more specific symbolic states.
initial_state = env.reset("train", 0)

# The robot instance from the environment
robot = env._pybullet_robot
# The PyBullet physics client ID
physics_client_id = env._physics_client_id

if not isinstance(robot, MobileSingleArmPyBulletRobot):
    logging.error("This test script is designed for a MobileSingleArmPyBulletRobot.")

# dyn = p.getDynamicsInfo(robot.robot_id, -1, physicsClientId=env._physics_client_id)
# print(f"\nMass, inertialFrame…{dyn}")
# input()


logging.info(f"Using robot: {robot.get_name()}")


#Store the robot's default arm and finger joint positions.
home_arm_joints = robot.initial_joint_positions

robot_obj = initial_state.get_objects(env._robot_type)[0]

#Store permament, fixed bodies
static_collision_bodies = get_all_non_robot_bodies(robot.robot_id, physics_client_id)

#print(f"Static_collision_bodies:{static_collision_bodies}")

#sys.exit(0)

#Define a rectangular workspace for base motion planning tests.
#(min_x, min_y, max_x, max_y)
workspace_bounds = (1.0, 0.2, 1.7, 1.3)

In [6]:
initial_base_pose = (0.4, 1.6, -np.pi/2)
target_base_pose = (0.75, 0.7441, np.pi / 4) # Target base pose
logging.info(f"Testing base-only motion from {initial_base_pose} to {target_base_pose}...")

#Resetting robot's state:
robot.move_base_to(initial_base_pose, physics_client_id)
robot.set_joints(home_arm_joints)

#Step simulation a bit to allow PyBullet to settle the state.
for _ in range(20):
    p.stepSimulation(physicsClientId=physics_client_id)

In [7]:
home_orn = env.get_robot_ee_home_orn()
# Keeping the z a bit high to avoid collision:
z = env.table_height + CFG.blocks_block_size/2 + 0.01
#logging.critical(f"Value of z: {z}.")
orn = (0, 0.7071, 0, 0.7071)

target_ee_pose = Pose(position=(1.5 , 0.75, z), 
                         orientation=home_orn)

coordinated_path = run_coordinated_motion_planning(
    robot,
    target_ee_pose=target_ee_pose,
    collision_bodies=static_collision_bodies,
    seed=CFG.seed,
    physics_client_id=physics_client_id,
    try_arm_only_first=True # Planner will try arm-only, fail, then try base+arm
)

base_path_waypoints, arm_path_waypoints = coordinated_path


2025-08-12 15:14:15 root [WARNING] Max time reached. No IKFast solution found.
2025-08-12 15:14:15 root [WARNING] No IK solutions found in 0.501 seconds


Coordinated Planning: Arm-only IK failed. Moving to base planning.


2025-08-12 15:14:15 root [WARNING] Max time reached. No IKFast solution found.
2025-08-12 15:14:15 root [WARNING] No IK solutions found in 0.501 seconds
2025-08-12 15:14:16 predicators.pybullet_helpers.motion_planning [WARNING] Planning arm motion to above target.
2025-08-12 15:14:16 predicators.pybullet_helpers.motion_planning [WARNING] Length of path to above target: 51.
2025-08-12 15:14:16 predicators.pybullet_helpers.motion_planning [WARNING] IK to move down succeeeded.
2025-08-12 15:14:16 predicators.pybullet_helpers.motion_planning [WARNING] Planning to move EE down to target.
2025-08-12 15:14:16 predicators.pybullet_helpers.motion_planning [WARNING] Length of path to move down to target: 61.


In [8]:
# def preprocess_path_for_differential_drive(  
#     raw_path: List[Tuple[float, float, float]],  
#     position_threshold: float = 0.05,  
#     rotation_threshold: float = 0.1  
# ) -> List[Tuple[float, float, float]]:  
#     """  
#     Preprocess BiRRT path to separate rotation and translation phases.  
#     This mimics iGibson's three-phase motion planning approach.  
#     """  
#     if len(raw_path) < 2:  
#         return raw_path  
      
#     processed_path = [raw_path[0]]  # Start with initial pose  
      
#     for i in range(1, len(raw_path)):  
#         prev_pose = processed_path[-1]  
#         curr_pose = raw_path[i]  
          
#         x1, y1, theta1 = prev_pose  
#         x2, y2, theta2 = curr_pose  
          
#         # Calculate position and orientation differences  
#         dx, dy = x2 - x1, y2 - y1  
#         position_dist = np.sqrt(dx*dx + dy*dy)  
          
#         # Calculate target heading for translation  
#         target_heading = np.arctan2(dy, dx) if position_dist > 0.01 else theta1  
          
#         # Circular difference for angles  
#         def circular_diff(a1, a2):  
#             diff = a1 - a2  
#             while diff > np.pi: diff -= 2 * np.pi  
#             while diff < -np.pi: diff += 2 * np.pi  
#             return diff  
          
#         heading_diff = abs(circular_diff(target_heading, theta1))  
#         final_diff = abs(circular_diff(theta2, target_heading))  
          
#         # Phase 1: Rotate to face target direction (if needed)  
#         if heading_diff > rotation_threshold and position_dist > position_threshold:  
#             processed_path.append((x1, y1, target_heading))  
          
#         # Phase 2: Translate while maintaining heading (if needed)  
#         if position_dist > position_threshold:  
#             processed_path.append((x2, y2, target_heading))  
          
#         # Phase 3: Rotate to final orientation (if needed)  
#         if final_diff > rotation_threshold:  
#             processed_path.append((x2, y2, theta2))  
#         elif position_dist <= position_threshold:  
#             # Just orientation change  
#             processed_path.append((x2, y2, theta2))  
      
#     return processed_path 

# base_path_waypoints = preprocess_path_for_differential_drive(raw_path=base_path_waypoints)

In [9]:
base_path_waypoints

[(0.4, 1.6, -1.5707963267948963),
 (0.4007662454414958, 1.6099946548731685, -1.567296694213553),
 (0.40153249088299164, 1.6199893097463371, -1.56379706163221),
 (0.4022987363244874, 1.6299839646195056, -1.560297429050867),
 (0.4030649817659832, 1.6399786194926742, -1.5567977964695239),
 (0.40383122720747905, 1.6499732743658426, -1.5532981638881806),
 (0.40459747264897483, 1.6599679292390113, -1.5497985313068376),
 (0.4053637180904706, 1.6699625841121797, -1.5462988987254946),
 (0.4061299635319664, 1.6799572389853483, -1.5427992661441514),
 (0.40689620897346224, 1.6899518938585167, -1.5392996335628082),
 (0.407662454414958, 1.6999465487316852, -1.5358000009814652),
 (0.41482689443022674, 1.7069913813115798, -1.5271351872229584),
 (0.42199133444549547, 1.7140362138914746, -1.5184703734644514),
 (0.4291557744607642, 1.7210810464713693, -1.5098055597059445),
 (0.436320214476033, 1.7281258790512641, -1.5011407459474377),
 (0.4434846544913017, 1.7351707116311588, -1.492475932188931),
 (0.450

In [10]:
def deduplicate_waypoints(waypoints, tol=1e-6):
    """
    Remove duplicate waypoints (anywhere in list, not just consecutive) 
    while preserving original order.
    """
    unique = []
    seen = set()
    for w in waypoints:
        # Round to tolerance to avoid floating-point issues
        key = tuple(round(v / tol) for v in w)
        if key not in seen:
            seen.add(key)
            unique.append(w)
    return unique

In [11]:
cleaned_base_path = deduplicate_waypoints(base_path_waypoints)

In [12]:
cleaned_base_path

[(0.4, 1.6, -1.5707963267948963),
 (0.4007662454414958, 1.6099946548731685, -1.567296694213553),
 (0.40153249088299164, 1.6199893097463371, -1.56379706163221),
 (0.4022987363244874, 1.6299839646195056, -1.560297429050867),
 (0.4030649817659832, 1.6399786194926742, -1.5567977964695239),
 (0.40383122720747905, 1.6499732743658426, -1.5532981638881806),
 (0.40459747264897483, 1.6599679292390113, -1.5497985313068376),
 (0.4053637180904706, 1.6699625841121797, -1.5462988987254946),
 (0.4061299635319664, 1.6799572389853483, -1.5427992661441514),
 (0.40689620897346224, 1.6899518938585167, -1.5392996335628082),
 (0.407662454414958, 1.6999465487316852, -1.5358000009814652),
 (0.41482689443022674, 1.7069913813115798, -1.5271351872229584),
 (0.42199133444549547, 1.7140362138914746, -1.5184703734644514),
 (0.4291557744607642, 1.7210810464713693, -1.5098055597059445),
 (0.436320214476033, 1.7281258790512641, -1.5011407459474377),
 (0.4434846544913017, 1.7351707116311588, -1.492475932188931),
 (0.450

In [13]:
def get_current_base_and_arm_pose(robot, state:State, objects: Sequence[Object], params: Array):

    current_base_pose = robot.get_base_pose(physics_client_id)
    current_joint_positions = robot.get_joints()

    return current_base_pose, current_joint_positions

target_base_pose = base_path_waypoints[-1]

move_option_memory = {}
param_space = Box(low=np.array([], dtype=np.float32),
                  high=np.array([], dtype=np.float32), dtype=np.float32)

move_option = create_move_base_option(robot, name="diff-drive",types=[env._robot_type], params_space=param_space,
                                    get_current_base_and_arm_pose=get_current_base_and_arm_pose, base_path=base_path_waypoints,
                                    target_base_pose = target_base_pose)


#The empty nd array is the empty param space that this option takes in as all
#the values are passed in to the option creation function. If this were to change,
#values will be passed in this array, and assigned to appropricate vars in _initialble
#and the param_space Box size will be changed accordingly.
grounded_move = move_option.ground([robot_obj], np.array([], dtype=np.float32))

In [14]:
assert grounded_move.initiable(move_option_memory)

print(grounded_move)

state = initial_state

while not grounded_move.terminal(move_option_memory):
    action = grounded_move.policy(move_option_memory)
    #logging.warning(f"\nNext action to be simulated:{action}.")
    #state = env.simulate(initial_state, action)

    #omega_r, omega_l = action.base_motion
    vel, omega = action.base_motion['params']

    # print(f"\nWheel velocities: {omega_r, omega_l}.")
    # input()

    #robot.set_wheel_motors(omega_r, omega_l, physics_client_id)
    # X, Y, THETA = robot.get_base_pose(physics_client_id)
    # vx, vy = 0.3 * np.cos(THETA), 0.3 * np.sin(THETA)
    # p.resetBaseVelocity(
    #     robot.robot_id,
    #     linearVelocity  = [vx, vy, 0.0],
    #     angularVelocity = [0.0, 0.0, omega],
    #     physicsClientId = physics_client_id)

    robot.set_wheel_motors(robot, vel, omega, physics_client_id)

    fixed_arm_pos = action.arr
    robot.set_motors(fixed_arm_pos)

    for _ in range(30):
        p.stepSimulation(physicsClientId=physics_client_id)
    time.sleep(0.05)

    # try:
    #     joint_solution = robot.inverse_kinematics(
    #             target_ee_pose, validate=True, set_joints=False)

    #     if joint_solution is not None:
    #         break

    # except:
    #     continue

    '''
    Now chain arm motion planning with this:
        - convert run_motion_planning to a singleParameterizedOption
        - connect to diff drive
    '''

# for waypoint in arm_path_waypoints:
#     action = Action(np.zeros(len(robot.action_space.low), dtype=float))
#     action._arr = waypoint

#     robot.set_motors(action.arr)
#     robot.set_wheel_motors(robot, 0.0, 0.0, physics_client_id)



print(f"\n Robot at {robot.get_base_pose(physics_client_id)} after executing differential drive.")

_Option(name='diff-drive', objects=[robby:robot], params=array([], dtype=float32))
[STEP 1] NAV - d=1.904 ptr=0 look=25 αW=2.63 v=0.05 ω=0.19
[STEP 2] NAV - d=1.904 ptr=0 look=25 αW=2.63 v=0.05 ω=0.19
[STEP 3] NAV - d=1.902 ptr=0 look=25 αW=2.63 v=0.05 ω=0.19
[STEP 4] NAV - d=1.899 ptr=0 look=25 αW=2.62 v=0.05 ω=0.20
[STEP 5] NAV - d=1.896 ptr=0 look=25 αW=2.61 v=0.05 ω=0.20
[STEP 6] NAV - d=1.894 ptr=0 look=25 αW=2.61 v=0.05 ω=0.20
[STEP 7] NAV - d=1.891 ptr=0 look=25 αW=2.60 v=0.05 ω=0.21
[STEP 8] NAV - d=1.888 ptr=0 look=25 αW=2.59 v=0.05 ω=0.21
[STEP 9] NAV - d=1.885 ptr=0 look=25 αW=2.58 v=0.05 ω=0.22
[STEP 10] NAV - d=1.882 ptr=0 look=25 αW=2.56 v=0.05 ω=0.23
[STEP 11] NAV - d=1.879 ptr=0 look=25 αW=2.55 v=0.05 ω=0.23
[STEP 12] NAV - d=1.876 ptr=0 look=25 αW=2.54 v=0.05 ω=0.24
[STEP 13] NAV - d=1.872 ptr=0 look=25 αW=2.52 v=0.05 ω=0.25
[STEP 14] NAV - d=1.868 ptr=0 look=25 αW=2.51 v=0.05 ω=0.25
[STEP 15] NAV - d=1.865 ptr=0 look=25 αW=2.49 v=0.05 ω=0.26
[STEP 16] NAV - d=1.861 pt

In [15]:
def analyze_debug(memory: Dict):
    trace = memory.get("debug_trace", [])
    if not trace:
        print("No data")
        return

    # 1) Are you ever hitting the deadband?
    dead_hits = [e for e in trace if e["deadband"]]
    print("Deadband hits:", len(dead_hits), "/", len(trace))

    # 2) Final few entries
    print("\nLAST 10 STEPS:")
    for e in trace[-10:]:
        print(f"d={e['dist']:.3f} br={e['branch']} ptr={e['path_ptr']}"
              f" look={e['look_idx']} α_W={e['alpha_W']:.2f}"
              f" α_G={e['alpha_G']:.2f} dead={e['deadband']}"
              f" v={e['v']:.2f} ω={e['ω']:.2f}")

In [16]:
analyze_debug(move_option_memory)

No data
